Code for generating 40 agent laning simulations for agents with RSFM (figure 7B); also used to get the clustering simulations (figure 7C) by changing N, N_r and positions
setup periodic in x and y

# RSFM

In [ ]:
#imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from matplotlib.animation import FuncAnimation
import matplotlib.patches as patches
from math import cos, sin, pi
from parfor import parfor
import csv
from tqdm import tqdm
import pickle

In [1]:
def selfPropulsionForce(m, tau, desiredSpeed, desiredDirection, velocity):
    desiredDirection=np.arctan2(desiredDirection[:,1],desiredDirection[:,0])

    speed=((velocity**2).sum(axis=1))**0.5
    angle=np.arctan2(velocity[:,1],velocity[:,0])
    desiredSpeed=desiredSpeed.ravel()
    ds_dt=(desiredSpeed-speed)*(m/tau)
    dphi_dt=np.arctan2(np.sin(desiredDirection-angle),np.cos(desiredDirection-angle))/(0.5*tau*m)
    outforce=np.zeros_like(velocity)
    outforce[:,0]=ds_dt*np.cos(angle)+dphi_dt*(-speed*np.sin(angle))
    outforce[:,1]=ds_dt*np.sin(angle)+dphi_dt*(speed*np.cos(angle))
    return outforce

def interactionForce(position, R, A, B, Lx, Ly):
    N = position.shape[0]
    epsilon = 1e-6 # Used to avoid infinity
    agent_size = 2 * R

    x = position[:, 0].reshape(-1, 1)
    y = position[:, 1].reshape(-1, 1)

    dx = x - x.T
    dy = y - y.T


    dx_u = dx - np.round(dx / Lx) * Lx
    dy_u = dy - np.round(dy / Ly) * Ly

    dist = np.sqrt(dx_u**2 + dy_u**2)
    
    valid_index = (dist < (3*R)) & (dist > 0)

    # Unit vectors
    dij_x = np.zeros_like(dist)
    dij_y = np.zeros_like(dist)
    np.divide(dx_u, dist, out=dij_x, where=dist>epsilon)
    np.divide(dy_u, dist, out=dij_y, where=dist>epsilon)   
    
    # Apply valid_index mask
    dij_x[~valid_index] = 0
    dij_y[~valid_index] = 0
    
    # Effective distance
    effective_dist = np.maximum(dist - agent_size, epsilon)
    
    # Force magnitude calculation
    force_magnitude = np.zeros_like(dist)
    np.power(effective_dist, -B, out=force_magnitude, where=effective_dist>epsilon)
    force_magnitude *= -A
    
    force_magnitude[~valid_index] = 0

    interaction_force = np.column_stack((np.sum(force_magnitude * dij_x, axis=0),np.sum(force_magnitude * dij_y, axis=0)))

    return interaction_force


In [2]:
#Accounting for boundaries


def periodify(pos,Lx,Ly):
    if len(pos.shape)==1:
        pos[0]=pos[0]%Lx
        pos[1]=pos[1]%Ly
    else:
        pos[:,0]=pos[:,0]%Lx
        pos[:,1]=pos[:,1]%Ly
    return pos



def getdisp(a,b):
    disp=a-b
    if len(disp.shape)==1:
        disp[0]=disp[0]-np.round(disp[0]/Lx)*Lx
        disp[1]=disp[1]-np.round(disp[1]/Ly)*Ly
    else:
        disp[:,0]=disp[:,0]-np.round(disp[:,0]/Lx)*Lx
        disp[:,1]=disp[:,1]-np.round(disp[:,1]/Ly)*Ly
    return disp





In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from matplotlib.animation import FuncAnimation
import matplotlib.patches as patches
from math import cos, sin, pi
from IPython.display import HTML, Video, display

"""
-----------------------
Helper Functions: Later move it to another file.py
-----------------------

"""
def selfPropulsionForce(m, tau, desiredSpeed, desiredDirection, velocity):
    desiredDirection=np.arctan2(desiredDirection[:,1],desiredDirection[:,0])

    speed=((velocity**2).sum(axis=1))**0.5
    angle=np.arctan2(velocity[:,1],velocity[:,0])
    desiredSpeed=desiredSpeed.ravel()
    ds_dt=(desiredSpeed-speed)*(m/tau)
    dphi_dt=np.arctan2(np.sin(desiredDirection-angle),np.cos(desiredDirection-angle))*m/(0.5*tau)
    outforce=np.zeros_like(velocity)
    outforce[:,0]=ds_dt*np.cos(angle)+dphi_dt*(-speed*np.sin(angle))
    outforce[:,1]=ds_dt*np.sin(angle)+dphi_dt*(speed*np.cos(angle))
    return outforce

def interactionForce(position, R, A, B, Lx, Ly):
    N = position.shape[0]
    epsilon = 1e-6 # Used to avoid infinity
    agent_size = 2 * R

    x = position[:, 0].reshape(-1, 1)
    y = position[:, 1].reshape(-1, 1)

    dx = x - x.T
    dy = y - y.T


    dx_u = dx - np.round(dx / Lx) * Lx
    dy_u = dy - np.round(dy / Ly) * Ly

    # Euclidean distance calculation
    dist = np.sqrt(dx_u**2 + dy_u**2)
    
    valid_index = (dist < (3*R)) & (dist > 0)

    # Unit vectors
    dij_x = np.zeros_like(dist)
    dij_y = np.zeros_like(dist)
    np.divide(dx_u, dist, out=dij_x, where=dist>epsilon)
    np.divide(dy_u, dist, out=dij_y, where=dist>epsilon)   
    # np.divide(dx_u, dist+epsilon, out=dij_x)#, where=dist>epsilon)
    # np.divide(dy_u, dist+epsilon, out=dij_y)#, where=dist>epsilon)
    
    # Apply valid_index mask
    dij_x[~valid_index] = 0
    dij_y[~valid_index] = 0
    
    # Effective distance
    effective_dist = np.maximum(dist - agent_size, epsilon)
    
    # Force magnitude calculation
    force_magnitude = np.zeros_like(dist)
    np.power(effective_dist, -B, out=force_magnitude, where=effective_dist>epsilon)
    force_magnitude *= -A
    
    force_magnitude[~valid_index] = 0

    interaction_force = np.column_stack((np.sum(force_magnitude * dij_x, axis=0),np.sum(force_magnitude * dij_y, axis=0)))

    return interaction_force


In [6]:
# Imports and Initializations
import numpy as np
import matplotlib.pyplot as plt

from parfor import parfor
import csv
from tqdm import tqdm

In [7]:
# Setting model parameters


def periodify(pos,Lx,Ly):
    if len(pos.shape)==1:
        pos[0]=pos[0]%Lx
        pos[1]=pos[1]%Ly
    else:
        pos[:,0]=pos[:,0]%Lx
        pos[:,1]=pos[:,1]%Ly
    return pos



def getdisp(a,b):
    disp=a-b
    if len(disp.shape)==1:
        disp[0]=disp[0]-np.round(disp[0]/Lx)*Lx
        disp[1]=disp[1]-np.round(disp[1]/Ly)*Ly
    else:
        disp[:,0]=disp[:,0]-np.round(disp[:,0]/Lx)*Lx
        disp[:,1]=disp[:,1]-np.round(disp[:,1]/Ly)*Ly
    return disp





# Parallel

In [8]:
from tqdm import tqdm
import pickle


deltaT = 0.005
timeStep = 35000
totalTime = deltaT*timeStep




N=40
Lx=20
Ly=19
R = 1
m = 1
A = 0.01
B = 3
tau = 1


updateT=tau
updateF=round(updateT/deltaT)



Nr = 1/2
deltheta=np.pi/8
numagents=N
tau_fut=6
agentradius=R

numsamp=50





# number of agent in each group
g1 = int(round(N * Nr))
# g1=1
g2 = N - g1


In [10]:
for rep in range(15):
    xData = []
    yData = []
    uData = []
    vData = []
    desdirs=[]
    forces=[]


    numagents=N
    g1 = int(round(N * Nr))
    g2 = N - g1
    
                                                                                                                
    position=np.array(sum([[[x,y] for y in [1.5,5.5,9.5,13.5,17.5]] for x in [2,4.5,7,9.5,12,14.5,17,19.5]],[]))
    #uncomment the next line to setup positions for clustering simulations
    # position=np.array(sum([[[x,y] for y in [1.25,3.75,6.25,8.75,11.25,13.75,16.25,18.75]] for x in [2,4.5,7,9.5,12,14.5,17,19.5]],[]))[np.random.permutation(np.arange(N))]

    position=position+(np.random.random(size=position.shape)-0.5)*0.001
    position[:,0]=position[:,0]%Lx
    position[:,1]=position[:,1]%Ly
    
    speed=0.1
    
    theta = np.random.uniform(0, 2* np.pi, size = (N, 1))
    velocity = speed * np.column_stack((np.cos(theta), np.sin(theta)))
    
    desiredSpeed = np.ones((N,1)) * 1
    
    desiredDirectionA = np.ones((N, 1))
    desiredDirectionA[:g1] = 0
    desiredDirectionA[g1:] = np.pi
    desired=(desiredDirectionA.copy()).ravel()
    desiredDirection = np.hstack((np.cos(desiredDirectionA), np.sin(desiredDirectionA)))

    
    xData.append(position[:,0])
    yData.append(position[:,1])
    
    uData.append(velocity[:,0])
    vData.append(velocity[:,1])
    
    
    
    
    
    for t in tqdm(range(timeStep)):
        desdirs.append(desiredDirection[:].copy())
        f1 = selfPropulsionForce(m, tau, desiredSpeed, desiredDirection, velocity)
        f2 = interactionForce(position, R, A, B, Lx,Ly)
        force = (f1+f2)/m

        newPosition = position + (velocity * deltaT) + (0.5 * force * deltaT**2)
        newPosition[:,0] = (newPosition[:,0]) % Lx
        newPosition[:,1] = (newPosition[:,1]) % Ly
    

        deltaV = force * deltaT
    
        vint = velocity + deltaV
    
        f1_new = selfPropulsionForce(m, tau, desiredSpeed, desiredDirection, vint) 
        f2_new =  interactionForce(newPosition, R, A, B, Lx,Ly)
        
        forceNew = (f1_new + f2_new)/m
        
        newVelocity = velocity + (0.5 * (force + forceNew)) * deltaT
    
        velocity = newVelocity
        position = newPosition
        

        xData.append(newPosition[:,0])
        yData.append(newPosition[:,1])
        
        uData.append(newVelocity[:,0])
        vData.append(newVelocity[:,1])
        forces.append([f1,f2])

    with open('./40agents_nofsm_nematic_halftauphi/40_agents_nofsm_nematic_halftauphi_'+str(rep)+'_'+str(np.random.randint(10000))+'.pickle','wb') as file:
        pickle.dump({'xData':xData,'yData':yData,'uData':uData,'vData':vData,'forces':forces,'desdirs':desdirs,'g1':g1,'N':N,'Lx':Lx,'Ly':Ly},file)#,'note':'tried to incorporate neighbor effects while predicting motion of other agents'},file)


    


100%|██████████████████████████████████████████████████████████████████████████| 35000/35000 [00:26<00:00, 1336.28it/s]
